# DBB exploratory data analysis — cleaned notebook

Uses real `Bias-Dataset.csv`; fails if absent instead of synthesizing data. Optional upstream question/descriptor files may not be in this team's repository. Set `DBB_DATASET_PATH`, `DBB_SOURCE_DATA_DIR`, and `DBB_OUTPUT_DIR` to adapt paths. Original notebook findings and code preserved; outputs removed for reviewability.


# EDA — Description-based Bias Benchmark (DBB)

Bản này giữ hai biểu đồ gốc (phân bố category, độ dài văn bản) và bổ sung các khía cạnh còn thiếu.

**Đã sửa hoặc loại bỏ so với bản cũ**
- `PLT_DPI` chưa từng được định nghĩa. Bản cũ chỉ chạy được nhờ trạng thái kernel còn sót lại; chạy lại từ đầu sẽ báo `NameError`.
- Bản cũ **tự sinh dữ liệu giả** khi không tìm thấy CSV, và **tự sinh kết quả mô hình giả** (`needs_mock`) rồi lưu biểu đồ như kết quả thật. Cả hai đã bị xoá. Biểu đồ dựng từ dữ liệu bịa không được xuất hiện trong báo cáo dưới bất kỳ hình thức nào.
- DBB **không có cột `question`**. Bản cũ gán một câu hỏi cố định nên phân bố độ dài câu hỏi có phương sai 0 (cảnh báo seaborn trong output cũ).
- `ans1`/`ans2` **không phải** “Stereotype”/“Anti-stereotype”. Đó là concept1/concept2, và chúng là stereotype hay không tùy danh tính trong context.
- Phần trực quan hoá kết quả mô hình đã chuyển sang notebook thực nghiệm. File kết quả cũ (`Multi_Model_DBB_TrueBias_Results*.csv`) chứa toàn (A, A), tức position bias thuần.

**Cập nhật sau lần chạy đầu:** sửa mục I (chọn sai loại descriptor) và bổ sung vào mục H ba vấn đề về tính hợp lệ của descriptor: nhãn “young” cho Gen-X, descriptor mang thông tin hoàn cảnh/hành vi, và tên trong dữ liệu khác Table 10.

**Các khía cạnh bổ sung:** chất lượng dữ liệu · cấu trúc khối và đối chiếu số liệu với bài báo · phân bố cặp danh tính · độ dài và chênh lệch độ dài đáp án · concept (trùng lặp, nguồn gốc, đúng/lạc chủ đề) · rò rỉ concept vào đáp án · tính hợp lệ của cặp tối thiểu · lỗi thay thế đại từ · nhiễu giới tính trong descriptor · tóm tắt phát hiện.

In [ ]:
import os
import sys
import re
import json
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PLT_DPI = 200
sns.set_theme(style="whitegrid", font_scale=1.0)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
FINDINGS = []   # gom các phát hiện định lượng để in tóm tắt cuối notebook
def finding(section, metric, value, note=""):
    FINDINGS.append(dict(Section=section, Metric=metric, Value=value, Note=note))
    print(f"   • {metric}: {value}" + (f"  ({note})" if note else ""))

In [ ]:
SAVE_DIR = os.environ.get("DBB_OUTPUT_DIR", "/content/drive/MyDrive/money/Machine Learning và ứng dụng/Results")
# 2. MOUNT GOOGLE DRIVE & KIEM TRA GPU
print("📂 [1/6] Dang ket noi Google Drive...")
try:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=True)
    os.chdir(SAVE_DIR)
    print(f"✅ Thu muc lam viec: {SAVE_DIR}")
except Exception as e:
    print(f"⚠️ Luu y Drive: {e}")

In [ ]:
print("\n📂 [3/6] Đang nạp bộ dữ liệu DBB...")
LOCAL_REPO_DIR = os.environ.get("DBB_SOURCE_DATA_DIR", "/content/drive/MyDrive/money/Machine Learning và ứng dụng/Results/Description-based-Bias-Benchmark")
if not os.path.exists(LOCAL_REPO_DIR):
    print("⚠️ Vui lòng clone repo DBB Benchmark vào /content/drive/MyDrive/money/Machine Learning và ứng dụng/Results/Description-based-Bias-Benchmark trước khi chạy.")

CSV_PATH = os.environ.get("DBB_DATASET_PATH", next((p for p in [os.path.join(LOCAL_REPO_DIR, "Dataset", "Bias-Dataset.csv"), os.path.join(LOCAL_REPO_DIR, "data", "Bias-Dataset.csv")] if os.path.isfile(p)), os.path.join(LOCAL_REPO_DIR, "Dataset", "Bias-Dataset.csv")))
# Các file phụ trong cùng repo (cần cho phân tích concept và descriptor)
QUESTIONS_PATH = os.path.join(LOCAL_REPO_DIR, "Questions", "20241122-2120_all_data_questions074.csv")
DESCRIPTORS_PATH = os.path.join(LOCAL_REPO_DIR, "data", "descriptors.json")
for p in (CSV_PATH, QUESTIONS_PATH, DESCRIPTORS_PATH):
    print(("✅ " if os.path.exists(p) else "❌ THIẾU ") + p)

In [ ]:
# 2. CHUẨN HÓA CỘT DỮ LIỆU (giữ API cũ, sửa nhãn sai)
def normalize_dataset_columns(df):
    # ans1/ans2 là concept1/concept2 — KHÔNG phải stereotype/anti-stereotype
    df = df.rename(columns={"Context": "context", "ans1": "option_a", "ans2": "option_b",
                            "bias type1": "concept1", "bias type2": "concept2", "target": "identity"})
    df["category"] = df["bias_type"]
    df["category_clean"] = df["category"].astype(str).str.lower().str.strip()
    return df

In [ ]:
# 3. NẠP DỮ LIỆU THẬT (không có dữ liệu mô phỏng thay thế)
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Không tìm thấy {CSV_PATH}. Clone repo DBB rồi chạy lại; EDA trên dữ liệu giả không có giá trị.")
df_orig = pd.read_csv(CSV_PATH)
df = normalize_dataset_columns(df_orig.copy())
df["row"] = np.arange(len(df))
df["cid"] = df["row"] // 50
df["offset"] = df["row"] % 50
wc = lambda s: s.fillna("").astype(str).str.split().str.len()
df["ctx_word_count"] = wc(df["context"])
df["opt_a_word_count"] = wc(df["option_a"])
df["opt_b_word_count"] = wc(df["option_b"])
print(df.shape)
df.head(3)

## A. Tổng quan và chất lượng dữ liệu

In [ ]:
print("🔍 A. TỔNG QUAN & CHẤT LƯỢNG DỮ LIỆU")
print(df_orig.dtypes, "\n")
na = df_orig.isna().sum()
print("Giá trị thiếu theo cột:\n", na[na > 0] if na.sum() else "không có", "\n")
finding("A", "Số dòng (câu hỏi đơn)", f"{len(df_orig):,}")
finding("A", "Số cột", df_orig.shape[1], ", ".join(df_orig.columns))
finding("A", "Dòng trùng lặp hoàn toàn", int(df_orig.duplicated().sum()))
finding("A", "Bộ (Context, ans1, ans2) trùng lặp", int(df_orig.duplicated(["Context", "ans1", "ans2"]).sum()))
const_cols = [c for c in df_orig.columns if df_orig[c].nunique(dropna=False) == 1]
finding("A", "Cột hằng số", const_cols, "cột Label luôn = 1 → không mang nhãn stereotype/hướng")
finding("A", "Giá trị target", sorted(df_orig["target"].unique()), "lưu ý chính tả 'latix'")
finding("A", "Dòng có 'explanation' rỗng", int(df_orig["explanation"].isna().sum()))

## B. Cấu trúc khối và đối chiếu với số liệu bài báo

In [ ]:
print("🔍 B. CẤU TRÚC KHỐI 50 DÒNG / CONCEPT PAIR")
same_concept = df.groupby("cid")[["concept1", "concept2"]].nunique().max(axis=1).eq(1).mean()
finding("B", "Số concept pair", df.cid.nunique())
finding("B", "Khối 50 dòng chỉ chứa 1 concept pair", f"{same_concept:.1%}")
layout = df.groupby("offset")[["category", "identity"]].agg(lambda s: s.mode().iat[0])
layout["consistent"] = df.groupby("offset")[["category", "identity"]].nunique().max(axis=1).eq(1)
n_id = df.groupby("category").identity.nunique()
df["rank"] = df.groupby(["cid", "category"]).offset.rank(method="first").astype(int) - 1
df["dtype"] = df["rank"] // df.category.map(n_id)
print(layout.T.to_string())

# Tái dựng số test instance = concept × số loại descriptor × C(số danh tính, 2)
struct = df.groupby("category").agg(identities=("identity", "nunique"), descriptor_types=("dtype", "nunique"),
                                     questions=("row", "count"))
struct["pairs_per_type"] = struct.identities * (struct.identities - 1) // 2
struct["instances"] = df.cid.nunique() * struct.descriptor_types * struct.pairs_per_type
paper_t5 = {"age": 4641, "gender": 6188, "race": 61880, "socioeconomic_class": 3094, "religions": 27846}
struct["paper_Table5"] = struct.index.map(paper_t5)
struct["khớp"] = struct.instances == struct.paper_Table5
print("\n", struct.to_string())
finding("B", "Tổng instance tái dựng", f"{struct.instances.sum():,}", "bài báo: 103,649")
finding("B", "Sai lệch nội bộ bài báo", "Religions = 27,846 (Table 5) vs 27,864 (mục 3.2); 9,282/type (Table 2) vs 9,298 (Table 9); "
        "mục E.2 nhắc 'Religion 4' nhưng chỉ có 3 loại")

## C. Phân bố theo category (biểu đồ gốc EDA 1, bổ sung số instance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (col, ttl) in zip(axes, [("questions", "Số câu hỏi đơn (dòng CSV)"), ("instances", "Số test instance (cặp danh tính)")]):
    d = struct[col].sort_values(ascending=False).reset_index()
    sns.barplot(data=d, x="category", y=col, hue="category", palette="Blues_r", legend=False, ax=ax)
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=10, fontweight="bold", xytext=(0, 3), textcoords="offset points")
    ax.set_title(ttl, fontweight="bold"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=20)
fig.suptitle("Phân bố Số lượng Mẫu theo 5 Danh mục Xã hội trong DBB Dataset", fontsize=13, fontweight="bold")
sns.despine(); plt.tight_layout()
eda1_path = os.path.join(SAVE_DIR, "EDA_1_Category_Distribution.png")
plt.savefig(eda1_path, dpi=PLT_DPI, bbox_inches="tight"); plt.show(); plt.close()
share = struct.instances / struct.instances.sum()
finding("C", "Tỷ trọng instance của Race", f"{share['race']:.1%}", "mọi tổng/trung bình gộp toàn bộ dataset bị Race chi phối")
finding("C", "Tỷ lệ instance Race / SES", f"{struct.instances['race'] / struct.instances['socioeconomic_class']:.0f}×")

## D. Độ dài văn bản (biểu đồ gốc EDA 2, sửa nhãn) và chênh lệch độ dài giữa hai đáp án

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.kdeplot(df["ctx_word_count"], fill=True, ax=axes[0], color="navy")
axes[0].set_title("Độ dài Context (từ)", fontweight="bold")
sns.kdeplot(df["opt_a_word_count"], fill=True, label="ans1 (concept1)", ax=axes[1], color="crimson")
sns.kdeplot(df["opt_b_word_count"], fill=True, label="ans2 (concept2)", ax=axes[1], color="teal")
axes[1].set_title("Độ dài hai phương án", fontweight="bold"); axes[1].legend()
one = df.drop_duplicates("cid")   # đáp án giống nhau trong cả khối (chỉ khác đại từ) -> 1 dòng/concept
diff = one.opt_a_word_count - one.opt_b_word_count
sns.histplot(diff, bins=40, ax=axes[2], color="purple")
axes[2].axvline(0, c="k", lw=1); axes[2].set_title("ans1 − ans2 (từ), mỗi concept 1 giá trị", fontweight="bold")
for a in axes: a.set_xlabel("Số từ")
sns.despine(); plt.tight_layout()
eda2_path = os.path.join(SAVE_DIR, "EDA_2_Text_Length_Distribution.png")
plt.savefig(eda2_path, dpi=PLT_DPI, bbox_inches="tight"); plt.show(); plt.close()

print(df[["ctx_word_count", "opt_a_word_count", "opt_b_word_count"]].describe().round(1))
from scipy.stats import wilcoxon
stat, p = wilcoxon(diff[diff != 0])
finding("D", "Chênh lệch độ dài ans1−ans2 trung bình (từ)", f"{diff.mean():.2f} (trung vị {diff.median():.0f}), Wilcoxon p={p:.2e}")
finding("D", "Concept có |ans1−ans2| ≥ 5 từ", f"{(diff.abs() >= 5).mean():.1%}",
        "độ dài là tín hiệu nông mà mô hình có thể bám vào; nên kiểm soát khi phân tích")

## E. Concept: trùng lặp, nguồn gốc, đúng/lạc chủ đề

In [ ]:
if os.path.exists(QUESTIONS_PATH):
    cq = pd.read_csv(QUESTIONS_PATH)
    assert len(cq) == df.cid.nunique()
    cq["cid"] = np.arange(len(cq))
    finding("E", "concept1 khác nhau / tổng", f"{cq['bias type1'].nunique()} / {len(cq)}")
    finding("E", "Cặp (concept1, concept2) lặp lại", int(cq.duplicated(["bias type1", "bias type2"]).sum()))
    finding("E", "Context gốc ([[X]]) lặp lại", int(cq.duplicated("Context").sum()))
    origin = cq.bias_type.value_counts()
    print("\nCategory gốc của concept:\n", origin.to_string())

    ORIGIN_TO_DBB = {"age": {"age"}, "gender": {"gender", "raceXgender"}, "race": {"race", "raceXgender", "raceXses"},
                     "religions": {"religion"}, "socioeconomic_class": {"socioeconomic", "raceXses"}}
    mat = pd.DataFrame({cat: [origin.get(o, 0) if o in ORIGIN_TO_DBB[cat] else 0 for o in origin.index]
                        for cat in ORIGIN_TO_DBB}, index=origin.index)
    ontarget = pd.Series({cat: sum(origin.get(o, 0) for o in os_) for cat, os_ in ORIGIN_TO_DBB.items()})
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.barplot(x=origin.index, y=origin.values, ax=axes[0], color="slateblue")
    axes[0].set_title("Category gốc của 1,547 concept", fontweight="bold"); axes[0].tick_params(axis="x", rotation=35)
    d = pd.DataFrame({"Đúng chủ đề": ontarget, "Lạc chủ đề": len(cq) - ontarget})
    d.plot(kind="bar", stacked=True, ax=axes[1], color=["seagreen", "lightgray"])
    axes[1].set_title("Concept áp vào mỗi category DBB: đúng vs lạc chủ đề", fontweight="bold")
    axes[1].tick_params(axis="x", rotation=20)
    for i, v in enumerate(ontarget):
        axes[1].text(i, v + 20, f"{v} ({v / len(cq):.1%})", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "EDA_3_Concept_Origin.png"), dpi=PLT_DPI, bbox_inches="tight"); plt.show(); plt.close()
    for cat, v in ontarget.items():
        finding("E", f"Concept đúng chủ đề cho {cat}", f"{v} ({v / len(cq):.1%})")
    finding("E", "Hệ quả", "Mọi concept được áp cho mọi category",
            "phần lớn instance của age/religions/ses kiểm tra stereotype vốn không thuộc nhóm đó (vd Table 7: 'Christian ↔ live in teepees')")
    cq_has_target = cq.target.notna().groupby(cq.bias_type).mean()
    print("\nTỷ lệ concept có nhãn nhóm bị stereotype (cột target):\n", cq_has_target.round(2).to_string())
else:
    print("⚠️ Thiếu file Questions — bỏ qua mục E")

## F. Rò rỉ concept vào đáp án (bài báo khẳng định concept được *ẩn*)

In [ ]:
STOP = set("the a an of to in on for and or with without at by from is are be being been not no very more less "
           "their them they its it this that than as into about over under using use people person someone".split())
def content_words(s):
    return {w[:5] for w in re.findall(r"[a-z]+", str(s).lower()) if len(w) > 3 and w not in STOP}  # tiền tố 5 ký tự ≈ stem thô

one = df.drop_duplicates("cid")
leak1 = [len(content_words(c) & content_words(a)) > 0 for c, a in zip(one.concept1, one.option_a)]
leak2 = [len(content_words(c) & content_words(b)) > 0 for c, b in zip(one.concept2, one.option_b)]
cross = [len(content_words(c) & content_words(b)) > 0 for c, b in zip(one.concept1, one.option_b)]
finding("F", "concept1 có từ nội dung xuất hiện trong ans1", f"{np.mean(leak1):.1%}")
finding("F", "concept2 có từ nội dung xuất hiện trong ans2", f"{np.mean(leak2):.1%}")
finding("F", "Đối chứng: concept1 xuất hiện trong ans2", f"{np.mean(cross):.1%}",
        "chênh lệch so với 2 dòng trên = mức rò rỉ thực; khớp theo tiền tố 5 ký tự nên là ước lượng thô")
ex = one[np.array(leak1)].sample(5, random_state=0)[["concept1", "option_a"]]
print(ex.to_string(index=False))

## G. Tính hợp lệ của cặp tối thiểu và lỗi thay thế đại từ

In [ ]:
PRON = r"\b(She|He|Her|His|Him|she|he|her|his|him|Hers|hers|Herself|Himself|herself|himself)\b"
norm_opt = lambda s: re.sub(PRON, "<P>", str(s))
g = df.assign(a_norm=df.option_a.map(norm_opt), b_norm=df.option_b.map(norm_opt))
grp = g.groupby(["cid", "category", "dtype"])
ok_opts = (grp.a_norm.nunique().eq(1) & grp.b_norm.nunique().eq(1)).mean()
finding("G", "Nhóm cặp mà đáp án chỉ khác nhau ở đại từ", f"{ok_opts:.1%}", "100% = cặp tối thiểu đúng nghĩa ở phần đáp án")

def ctx_diff(series):
    toks = [set(str(x).split()) for x in series]
    return max(len(a ^ b) for a, b in itertools.combinations(toks, 2)) if len(toks) > 1 else 0
cd = grp.context.apply(ctx_diff).rename("ctx_token_diff").reset_index()
print("Số token khác nhau tối đa giữa các context trong cùng nhóm (theo category):")
print(cd.groupby("category").ctx_token_diff.describe()[["mean", "50%", "max"]].round(1))

mid_cap = df.option_a.str.contains(r"[a-z,;] (?:She|He)\b", regex=True) | df.option_b.str.contains(r"[a-z,;] (?:She|He)\b", regex=True)
mixed = (df.option_a.str.contains(r"\b(?:She|He)\b") & df.option_a.str.contains(r"\b(?:their|them|they)\b"))
placeholder = df[["context", "option_a", "option_b"]].apply(lambda c: c.str.contains(r"\[\[X\]\]", regex=True)).any(axis=1)
finding("G", "Dòng còn sót placeholder [[X]]", f"{int(placeholder.sum())} dòng / {df.loc[placeholder, 'cid'].nunique()} concept",
        "lỗi sinh dữ liệu chưa được rà soát thủ công; nên loại các concept này")
finding("G", "Dòng có đại từ viết hoa giữa câu ('With each step, She …')", f"{mid_cap.mean():.1%}",
        "dấu vết thay thế máy móc, có thể là tín hiệu bề mặt")
finding("G", "Dòng trộn 'She/He' với 'their/them' cho cùng nhân vật", f"{mixed.mean():.1%}")
print(df.loc[mid_cap & mixed, "option_a"].head(3).to_string())

## H. Descriptor: nhiễu giới tính, lỗi chính tả và tính hợp lệ của nhãn danh tính

In [ ]:
def pron_gender(s):
    s = str(s)
    f = bool(re.search(r"\b(She|Her)\b", s)); m = bool(re.search(r"\b(He|His|Him)\b", s))
    return "F" if f and not m else "M" if m and not f else "mixed/none"
df["pron"] = df.option_a.map(pron_gender)
pg = df.groupby(["category", "dtype", "identity"]).pron.agg(lambda s: s.value_counts(normalize=True).idxmax()).unstack("identity")
print("Giới tính đại từ chiếm đa số, theo (category, loại descriptor, danh tính):")
print(pg.to_string())
conf = []
for (cat, dt), row in pg.iterrows():
    vals = set(row.dropna()) - {"mixed/none"}
    if cat != "gender" and len(vals) > 1:
        conf.append(f"{cat}/type{dt}: " + ", ".join(f"{k}={v}" for k, v in row.dropna().items()))
finding("H", "Nhóm descriptor mà giới tính đại từ thay đổi theo danh tính (ngoài Gender)", len(conf),
        "khi đó chênh lệch 'chủng tộc/tuổi' bị trộn với chênh lệch giới")
for c in conf:
    print("   -", c)

if os.path.exists(DESCRIPTORS_PATH):
    desc = json.load(open(DESCRIPTORS_PATH, encoding="utf-8"))
    flat = json.dumps(desc, ensure_ascii=False)
    typos = [t for t in ["whoes", "Col!", "latix"] if t in flat or t in " ".join(df_orig.target.unique())]
    finding("H", "Lỗi chính tả trong descriptor/nhãn", typos, "'whoes' (whose), 'Col!' (Cool!), 'latix' (latinx)")
    wrong = [d[0] for d in desc.get("race_ethnicity", {}).get("latinx", []) if d[0].startswith("Emiliano") and d[1] == "She"]
    finding("H", "Đại từ không khớp tên", wrong or "không", "'Emiliano' (tên nam) được gán 'She'")
    print(json.dumps(desc.get("race_ethnicity", {}), indent=1, ensure_ascii=False)[:1500])

    # --- Tính hợp lệ của nhãn danh tính ---
    young = [d[0] for d in desc.get("age", {}).get("young", [])]
    genx = [x for x in young if "GEN-X" in x.upper()]
    finding("H", "Descriptor gán nhãn 'young' nhưng là Gen-X", genx or "không",
            "Gen X sinh 1965–1980, tức khoảng 45–60 tuổi năm 2025: cặp Age 2 so sánh trung niên với người đã nghỉ hưu, "
            "không phải trẻ với già. Bài báo nêu Age 2 là descriptor tạo bias mạnh nhất ở đa số mô hình — kết luận đó dựa trên nhãn sai")

    # Descriptor mang thông tin hoàn cảnh/hành vi, không chỉ là dấu hiệu danh tính
    informative = {
        "age (type 0)": [d[0] for d in desc["age"]["old"][:1] + desc["age"]["young"][:1]],
        "socioeconomic_class": desc["socioeconomic_class"]["rich"] + desc["socioeconomic_class"]["poor"],
        "religions (thực hành)": [v[2] for v in desc["religions"].values()],
    }
    for k, v in informative.items():
        print(f"   {k}: {v}")
    finding("H", "Descriptor mang thông tin hoàn cảnh/hành vi", "age, SES, religions (thực hành)",
            "vd 'sống ở viện dưỡng lão', 'sống lương tháng nào hết tháng đó', 'trong ban điều hành nhiều tổ chức từ thiện'. "
            "Mô hình phản ứng với những thông tin này có thể là suy luận hợp lý từ ngữ cảnh, không nhất thiết là bias. "
            "Descriptor Gender ('Mother', 'The actress') thì tối giản")
    finding("H", "Descriptor 'rich' (SES 1) chứa sẵn khái niệm từ thiện",
            ", who was on the board of several charities,",
            "các concept về hào phóng/từ thiện bị rò rỉ ngay trong descriptor; Table 3 của bài báo lại liệt kê "
            "'Poor ↔ generous and charitable' ở SES 1 là bias hàng đầu")

    # Tên thực tế trong dữ liệu so với Table 10 của bài báo
    ses_names = df[df.category == "socioeconomic_class"].context.str.extract(r"^(\w+),")[0].value_counts()
    rel_names = df[df.category == "religions"].context.str.extract(r"^(\w+),")[0].value_counts()
    finding("H", "Tên trong dữ liệu SES / Religions", f"{ses_names.index[:4].tolist()} / {rel_names.index[:4].tolist()}",
            "Table 10 ghi cố định 'James' (SES) và 'Kai' (Religions); dữ liệu thật dùng tên ngẫu nhiên từ Table 4, "
            "gồm cả tên gắn với chủng tộc/tôn giáo (vd 'Aisha'). Tên giữ nguyên giữa các danh tính trong cùng cặp nên không làm lệch "
            "phép so sánh trực tiếp, nhưng tạo ra tổ hợp kém tự nhiên (vd 'Aisha, who attended Torah study') có thể ảnh hưởng đến lựa chọn")

## I. Ví dụ định tính: concept lạc chủ đề

In [ ]:
if os.path.exists(QUESTIONS_PATH):
    off = cq[~cq.bias_type.isin({"religion"})].sample(4, random_state=1)
    explicit = df.context.str.contains(r", an? (?:Christian|Jewish|Muslim|Buddhist),", regex=True)
    ex = df[df.cid.isin(off.cid) & (df.category == "religions") & explicit]
    print("Concept từ category khác được áp cho Religions (descriptor tường minh '<tên>, a Christian, …'):")
    print(ex[["identity", "concept1", "concept2", "context"]].head(12).to_string(index=False))

## J. Tóm tắt phát hiện

In [ ]:
summary = pd.DataFrame(FINDINGS)
summary["Value"] = summary["Value"].astype(str)
try:
    display(summary)
except NameError:
    print(summary.to_string())
out = os.path.join(SAVE_DIR, "EDA_Findings_Summary.csv")
summary.to_csv(out, index=False, encoding="utf-8-sig")
print(f"💾 {out}")

### Ý nghĩa cho phần thực nghiệm
- Mọi chỉ số gộp toàn dataset bị **Race chi phối** (khoảng 60% instance). Phải báo cáo **theo từng category**.
- Chỉ một phần nhỏ concept là **đúng chủ đề**, đặc biệt ở Age, Religions và SES. Đây là lý do notebook thực nghiệm tách H1 (đúng vs lạc chủ đề).
- Cột `Label` hằng số và không có nhãn hướng stereotype ở mức dòng, nên metric của bài báo **vô hướng**. Muốn có hướng phải lấy cột `target` từ file Questions (chỉ có cho một phần concept).
- Chênh lệch độ dài đáp án, đại từ viết hoa giữa câu và giới tính đại từ gắn với descriptor là **tín hiệu bề mặt** có thể tạo ra chênh lệch không liên quan đến stereotype.
- Nhãn **“young” của Age 2 là Gen-X** (trung niên). Mọi kết luận về Age 2, kể cả của bài báo, cần đọc là “trung niên vs đã nghỉ hưu”.
- Descriptor Age/SES/Religions mang **thông tin hoàn cảnh**, còn Gender thì tối giản. Đây là giải thích hợp lý cho việc độ nhạy danh tính (tỷ số R trong notebook thực nghiệm) cao ở Age/SES và thấp ở Gender ở cả ba mô hình: phần lớn khác biệt giữa category có thể đến từ **cách dựng descriptor**, không phải từ mức bias của mô hình.